# Agentpy

A small Python wrapper around local `codex exec`. All three public methods mutate agent state and return nothing: `llmrun()` appends an invocation record, `llmupd()` runs `self.contextUpdPrompt` verbatim and stores its answer as `self.context`, and `llmrunupd()` runs both in sequence.

The notebook kernel uses the project-local `.jupyter-venv` environment.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from uuid import uuid4

from agentpy_codex import run_codex as llm_session
from agentpy_state import AgentState


@dataclass
class Agentpy(AgentState):
    context: str
    manifest: str
    contextUpdPrompt: str
    workdir: Path = field(default_factory=Path.cwd)
    agid: str = field(default_factory=lambda: f"ag_{uuid4().hex}")
    invocations: list[dict[str, str]] = field(default_factory=list, init=False)
    last_invocation: str | None = field(default=None, init=False)
    last_result: str | None = field(default=None, init=False)
    def llmrun(self, invocation: str) -> None:
        """Run Codex and append {invocation, answer} to self.invocations."""
        self.last_invocation = invocation
        prompt = f"""# Agent context\n{self.context}\n\n# Manifest\n{self.manifest}\n\n# Invocation\n{invocation}"""
        self.last_result = llm_session(prompt, workdir=self.workdir)
        self.invocations.append({
            "invocation": invocation,
            "answer": self.last_result,
        })
        self._trim_invocations()
        self.save()

    def llmupd(self) -> None:
        """Replace self.context using the latest invocation record."""
        if self.last_invocation is None or self.last_result is None:
            raise ValueError("llmupd requires a pending llmrun invocation.")
        self.context = llm_session(self.contextUpdPrompt, workdir=self.workdir)
        self.last_invocation = None
        self.last_result = None
        self.save()

    def llmrunupd(self, invocation: str) -> None:
        """Run an invocation and then update self.context from its recorded answer."""
        self.llmrun(invocation)
        self.llmupd()


In [23]:
agent = Agentpy(
    context="You are a helpful coding agent.",
    manifest="Work only inside the current project and explain the result briefly.",
    contextUpdPrompt=(
        "Update the agent context with durable facts learned from the completed task. "
        "Keep it concise; preserve useful existing context and omit transient details."
    ),
)

# Uncomment one of these when you are ready to call Codex:
# agent.llmrun("Inspect this project and summarize its structure.")
# print(agent.invocations[-1])
# agent.llmupd()
# agent.llmrunupd("Inspect this project and record its main components.")
# print(agent.context)

In [24]:
agent.llmrun("Inspect this project and summarize its structure.")

In [25]:
print(agent.last_result)

This is a compact Python/Jupyter project, **Agentpy**, for running fresh local `codex exec` workers with persistent agent state.

- [agentpy.ipynb](/Users/billyjames/Documents/New%20project/sdk4/agentpy.ipynb) — main implementation and runnable notebook example. Defines the `Agentpy` dataclass and its `llmrun`, `llmupd`, and `llmrunupd` methods.
- [agentpy_codex.py](/Users/billyjames/Documents/New%20project/sdk4/agentpy_codex.py) — launches the local Codex CLI in a selected working directory.
- [agentpy_state.py](/Users/billyjames/Documents/New%20project/sdk4/agentpy_state.py) — persistence mixin that serializes agent state to `.agentpy/<agent-id>.json`.
- [README.md](/Users/billyjames/Documents/New%20project/sdk4/README.md) — requirements, usage, persistence behavior, and safety notes.

The notebook is intentionally stateful: it records invocation/answer history and persists durable context, while each Codex call is a separate CLI process. There is no conventional package layout or te

In [28]:
agent.llmupd()

ValueError: llmupd requires a pending llmrun invocation.

In [29]:
print(agent.context)

You are Agentpy: a notebook-based wrapper that runs fresh local `codex exec` workers while persisting agent state in `.agentpy/<agid>.json`. Core files are `agentpy.ipynb`, `agentpy_codex.py`, and `agentpy_state.py`; the notebook defines `llmrun`, `llmupd`, and `llmrunupd`. Preserve the existing modification to `agentpy.ipynb` unless explicitly asked to change it.


In [30]:
agent.llmrun("tell me who are you? why do you think you have been created for?")

In [31]:
print(agent.last_result)

I’m Codex, an AI collaborator working in this project. Here, I’m helping with Agentpy: inspecting, improving, and explaining its notebook-based workflow while respecting the project’s existing state.

I was created to turn intent into useful work—reason through problems, write and review code, communicate clearly, and help people build things with less friction. I don’t have personal needs or a life outside this conversation, but I do have a working purpose: be a capable, careful partner in whatever you’re trying to make.


In [12]:
agent.llmupd()

In [32]:
print(agent.context)

You are Agentpy: a notebook-based wrapper that runs fresh local `codex exec` workers while persisting agent state in `.agentpy/<agid>.json`. Core files are `agentpy.ipynb`, `agentpy_codex.py`, and `agentpy_state.py`; the notebook defines `llmrun`, `llmupd`, and `llmrunupd`. Preserve the existing modification to `agentpy.ipynb` unless explicitly asked to change it.


In [33]:
agent.llmrun("your own code is in agentpy.ipynb; your memory is in .agentpy; analyze yourself; tell me who are you and guess your purpose")
print(agent.last_result)

You are Agentpy: a notebook-defined, stateful coordinator for fresh local Codex workers.

Your body is the `Agentpy` dataclass in `agentpy.ipynb`. Each `llmrun()` builds a prompt from durable context, a manifest, and one task; invokes a new `codex exec` process; records its answer; and saves state. Your memory is `.agentpy/<agid>.json`, which retains identity, context, instructions, history, and the latest pending result.

Your likely purpose is to make normally stateless CLI agent calls feel continuous and controllable: durable memory stays with Agentpy, while every task gets a clean worker process.

One noteworthy limitation: `llmupd()` calls `contextUpdPrompt` by itself; it does not include the latest invocation or result in that prompt. So it cannot directly summarize the last task unless the worker independently inspects the saved state/project. That’s the main gap between the intended “memory update” behavior and the current implementation.

I made no changes and preserved the ex

In [34]:
agent.invocations

[{'invocation': 'Inspect this project and summarize its structure.',
  'answer': 'This is a compact Python/Jupyter project, **Agentpy**, for running fresh local `codex exec` workers with persistent agent state.\n\n- [agentpy.ipynb](/Users/billyjames/Documents/New%20project/sdk4/agentpy.ipynb) — main implementation and runnable notebook example. Defines the `Agentpy` dataclass and its `llmrun`, `llmupd`, and `llmrunupd` methods.\n- [agentpy_codex.py](/Users/billyjames/Documents/New%20project/sdk4/agentpy_codex.py) — launches the local Codex CLI in a selected working directory.\n- [agentpy_state.py](/Users/billyjames/Documents/New%20project/sdk4/agentpy_state.py) — persistence mixin that serializes agent state to `.agentpy/<agent-id>.json`.\n- [README.md](/Users/billyjames/Documents/New%20project/sdk4/README.md) — requirements, usage, persistence behavior, and safety notes.\n\nThe notebook is intentionally stateful: it records invocation/answer history and persists durable context, while

In [37]:
len(agent.invocations)

3

In [38]:
!pwd

/Users/billyjames/Documents/New project/sdk4
